# Descriptive analysis
This script provides descriptive analysis of the citation network.

## Setup

In [1]:
import pandas as pd
from collections import defaultdict

data_dir = r"C:\Users\Popov\Documents\Studies\IMT_studies\Projects\Econ_2\Datasets"
export_dir = data_dir

patents      = pd.read_stata(data_dir + r"\patents.dta")
citations    = pd.read_stata(data_dir + r"\citations.dta")
cited_owners = pd.read_stata(data_dir + r"\cited_owners.dta")
mnes         = pd.read_stata(r"C:\Users\Popov\Documents\Studies\IMT_studies\Projects\Econ_2\Italian_affiliates_MNEs_2024.dta")

print("patents:", patents.shape, "| citations:", citations.shape, "| cited_owners:", cited_owners.shape)

patents: (629225, 10) | citations: (1809184, 2) | cited_owners: (559652, 2)


## Group membership
Each firm's network = its parent + all co-affiliates sharing that parent. Asterisks stripped so ids match the patent applicant format.

In [2]:
mnes["subs_key"]   = mnes["subs_id"].str.replace("*","",regex=False).str.strip().str.upper()
mnes["parent_key"] = mnes["parent_id"].str.replace("*","",regex=False).str.strip().str.upper()

group_members = defaultdict(set)
for _, r in mnes.iterrows():
    group_members[r["parent_key"]].add(r["subs_key"])
    group_members[r["parent_key"]].add(r["parent_key"])

firm_group = {}
for _, r in mnes.iterrows():
    firm_group[r["subs_key"]] = group_members[r["parent_key"]]

print("firms mapped to a group:", len(firm_group))

firms mapped to a group: 37263


Overall number of Italian firms in the dataset.

## Owner lookup
Combine the cited-owner pull with our own patents' owners. A cited patent can have several owners, so keep pub-owner PAIRS, not a one-owner dict.

In [3]:
cited_owners.columns = ["pub", "owner_key"]

own = (patents.dropna(subset=["publication_number"])[["publication_number", "bvd_key"]]
       .rename(columns={"publication_number": "pub", "bvd_key": "owner_key"}))

owners = pd.concat([own, cited_owners], ignore_index=True).dropna().drop_duplicates()
print("pub-owner pairs:", len(owners),
      "| distinct patents with owner:", owners["pub"].nunique())

pub-owner pairs: 1143177 | distinct patents with owner: 1094094


1,143,177 pairs across 1,094,094 distinct patents means about 49,000 patents have more than one owner

## In-network flag
A citation is in-network if ANY owner of the cited patent is in the citing firm's group, excluding the firm itself.

In [4]:
# citing firm for each link (citing patents are always ours)
pub_to_citer = (patents.dropna(subset=["publication_number"])
                .set_index("publication_number")["bvd_key"].to_dict())
citations = citations.reset_index(drop=True)
citations["citing_owner"] = citations["publication_number"].map(pub_to_citer)
citations["link_id"] = citations.index

# explode: one row per (citation link x owner of the cited patent)
link = citations.merge(owners, left_on="backward_citations", right_on="pub", how="left")
print("links after owner merge:", len(link))
print("links with a known cited owner:", f'{link["owner_key"].notna().mean():.1%}')

links after owner merge: 1874899
links with a known cited owner: 75.8%


An in-network link can be either of two things, since the group is parent + co-affiliates:

affiliate -> parent — the affiliate cites a patent owned by its own parent


affiliate -> sibling — the affiliate cites a patent owned by another affiliate under the same parent

In [5]:
link["in_net"] = [
    (pd.notna(c) and pd.notna(o) and o in firm_group.get(c, ()) and o != c)
    for c, o in zip(link["citing_owner"], link["owner_key"])
]
link["is_self"] = [
    (pd.notna(c) and pd.notna(o) and o == c)
    for c, o in zip(link["citing_owner"], link["owner_key"])
]

# any owner in-network -> the citation link is in-network
citations["network_cite"] = citations["link_id"].map(
    link.groupby("link_id")["in_net"].max()).fillna(False)
citations["self_cite"] = citations["link_id"].map(
    link.groupby("link_id")["is_self"].max()).fillna(False)
citations["attributable"] = citations["link_id"].map(
    link.groupby("link_id")["owner_key"].count().gt(0)).fillna(False)

print("total links:", len(citations))
print("attributable links:", citations["attributable"].sum())
print("in-network links (excl. self):", citations["network_cite"].sum())
print("self-citation links:", citations["self_cite"].sum())

total links: 1809184
attributable links: 1355766
in-network links (excl. self): 3417
self-citation links: 144450


"Attributable" means: we know who owns the cited patent.

In [6]:
# restrict to citations by Italian affiliates in the MNE list
subs_set = set(mnes["subs_key"])
cit_aff = citations[citations["citing_owner"].isin(subs_set)]

print("links by Italian affiliates:", len(cit_aff))
print("attributable links:", cit_aff["attributable"].sum())
print("in-network links (excl. self):", cit_aff["network_cite"].sum())
print("self-citation links:", cit_aff["self_cite"].sum())

links by Italian affiliates: 785403
attributable links: 608088
in-network links (excl. self): 3417
self-citation links: 66289


## Split by ownership
The citing firm is always an Italian affiliate; `parent_is_foreign` describes whether its ultimate owner sits abroad.

In [7]:
# flag: is the citing affiliate's ultimate owner foreign?
foreign_map = mnes.set_index("subs_key")["foreign"].to_dict()
citations["parent_is_foreign"] = citations["citing_owner"].map(foreign_map)

split = (citations.groupby("parent_is_foreign")
         .agg(attributable  = ("attributable",  "sum"),
              in_network    = ("network_cite",  "sum"))
         .rename(index={0.0: "Italian parent", 1.0: "Foreign parent"}))

split["in_network_pct"] = (100 * split["in_network"] / split["attributable"]).round(3)

print("Citations by Italian affiliates, split by their parent's nationality:")
print(split)

Citations by Italian affiliates, split by their parent's nationality:
                   attributable  in_network  in_network_pct
parent_is_foreign                                          
Italian parent           260255        1681           0.646
Foreign parent           347833        1736           0.499


## Firm-level network-citation share
Restricted to Italian affiliates in the MNE list — only they have a group, so only they can have in-network citations. Network = own group (parent + co-affiliates), self-citations excluded.

In [8]:
# citing firms that are Italian affiliates in our list (others have no group)
subs_set = set(mnes["subs_key"])
cit = citations[citations["citing_owner"].isin(subs_set)].copy()

firm_cites = cit.groupby("citing_owner").agg(
    total_cites   = ("backward_citations", "size"),
    network_cites = ("network_cite", "sum"),
).reset_index().rename(columns={"citing_owner": "bvd_key"})

firm_cites["network_share"] = firm_cites["network_cites"] / firm_cites["total_cites"]

print("Italian affiliates in dataset:      ", len(subs_set))
print("...with >=1 backward citation:      ", len(firm_cites))
print("...with >=1 in-network citation:    ", (firm_cites["network_cites"] > 0).sum())
print("share of citing affiliates w/ signal:",
      f'{(firm_cites["network_cites"] > 0).mean():.1%}')

Italian affiliates in dataset:       37263
...with >=1 backward citation:       2569
...with >=1 in-network citation:     181
share of citing affiliates w/ signal: 7.0%


## Distribution of the share
How large the share is among firms that have any in-network citation - this is the identifying variation.

In [9]:
nz = firm_cites[firm_cites["network_cites"] > 0]

print("network_share:")
print(nz["network_share"].describe())
print()
print("network_cites per firm:")
print(nz["network_cites"].describe())

network_share:
count    181.000000
mean       0.064471
std        0.113179
min        0.000202
25%        0.008333
50%        0.024194
75%        0.071429
max        1.000000
Name: network_share, dtype: float64

network_cites per firm:
count    181.000000
mean      18.878453
std       61.741276
min        1.000000
25%        1.000000
50%        3.000000
75%       10.000000
max      600.000000
Name: network_cites, dtype: float64


## Effective sample
The funnel from all affiliates down to those carrying in-network signal.

In [10]:
n_aff     = len(subs_set)                                    # all Italian affiliates
n_citing  = len(firm_cites)                                  # ...with >=1 backward citation
n_signal  = (firm_cites["network_cites"] > 0).sum()          # ...with >=1 in-network citation

print(f"Italian affiliates in dataset:   {n_aff:>7,}")
print(f"...with >=1 backward citation:   {n_citing:>7,}  ({n_citing/n_aff:.1%} of affiliates)")
print(f"...with >=1 in-network citation: {n_signal:>7,}  ({n_signal/n_citing:.1%} of citing affiliates)")

# split the firms that carry signal by parent nationality
firm_cites["parent_is_foreign"] = firm_cites["bvd_key"].map(
    mnes.set_index("subs_key")["foreign"].to_dict())

by_parent = (firm_cites.assign(has_signal=firm_cites["network_cites"] > 0)
             .groupby("parent_is_foreign")
             .agg(citing_firms=("bvd_key", "size"),
                  with_signal=("has_signal", "sum"))
             .rename(index={0.0: "Italian parent", 1.0: "Foreign parent"}))
by_parent["pct"] = (100 * by_parent["with_signal"] / by_parent["citing_firms"]).round(1)

print()
print("Affiliates carrying in-network signal, by parent nationality:")
print(by_parent)

Italian affiliates in dataset:    37,263
...with >=1 backward citation:     2,569  (6.9% of affiliates)
...with >=1 in-network citation:     181  (7.0% of citing affiliates)

Affiliates carrying in-network signal, by parent nationality:
                   citing_firms  with_signal  pct
parent_is_foreign                                
Italian parent             1281          102  8.0
Foreign parent             1288           79  6.1


In [11]:
# how many co-affiliates does each signal firm's group have?
grp_size = mnes.groupby("parent_key").size()
firm_cites["group_size"] = firm_cites["bvd_key"].map(
    mnes.set_index("subs_key")["parent_key"]).map(grp_size)

print(firm_cites.groupby(firm_cites["network_cites"] > 0)["group_size"].describe())

                count      mean        std  min  25%  50%  75%    max
network_cites                                                        
False          2388.0  7.088358  39.866537  1.0  1.0  2.0  4.0  642.0
True            181.0  4.033149   4.838389  1.0  1.0  2.0  4.0   33.0


Group size does not drive the signal. Affiliates with in-network citations sit in slightly
smaller groups on average (4.0 vs 7.1 co-affiliates), with identical medians (2.0) and
quartiles — the gap comes entirely from the upper tail, where the largest groups (up to 642
affiliates) show no in-network citations at all. Scarcity therefore reflects technological
distance within groups rather than a lack of potential co-affiliates to cite.

## Save
Keyed on `bvd_key`, to merge into the firm-year panel later.

In [12]:
firm_cites.to_stata(export_dir + r"\network_share.dta",
                    write_index=False, version=118)
print("saved network_share.dta")

saved network_share.dta
